<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-ComputerVision/blob/main/DeepLearning_Segmentierung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep-Learning-Segmentierung in Aktion

**Computer Vision – BSc Business Artificial Intelligence – FHNW**

In diesem Notebook probierst du drei Deep-Learning-Ansätze für Segmentierung aus, die wir in der Session besprochen haben. Du brauchst die Modelle nicht selbst zu trainieren – wir nutzen vortrainierte Netze und lassen sie auf Bildern Vorhersagen machen.

## Drei Methoden, drei Output-Typen

| Methode | Architektur-Familie | Output |
|---|---|---|
| **FCN-ResNet50** | Fully Convolutional Network | Semantic Mask – Klasse pro Pixel |
| **Mask R-CNN** | Two-Stage Detector + Mask Head | Eine Maske **pro Instanz** |
| **SAM** | Foundation Model (promptbar) | Maske rund um deinen Klick |

## Lernziele

Nach diesem Notebook kannst du:
- vortrainierte Segmentierungsmodelle aus PyTorch und Hugging Face laden und anwenden
- den Output-Unterschied zwischen Semantic, Instance und promptbarer Segmentierung an konkreten Bildern erkennen
- die Stärken und Grenzen der drei Ansätze für eigene Bilder einschätzen

> 💡 **Vor dem Start:** Aktiviere in Colab die GPU unter *Laufzeit → Laufzeittyp ändern → T4 GPU*. Die CPU funktioniert auch, ist aber bei SAM deutlich langsamer (ca. 30–60s pro Bild).

## 1. Setup

Die meisten Pakete sind in Colab schon installiert. `transformers` brauchen wir für SAM.

In [ ]:
# Falls nötig: transformers aktualisieren (auf Colab i.d.R. nicht zwingend)
# !pip install -q --upgrade transformers

import io
import requests
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torchvision

# Umgebung erkennen
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# GPU verfügbar?
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("Hinweis: Ohne GPU läuft besonders SAM langsam. Reicht für dieses Notebook trotzdem.")

### Hilfsfunktionen für Visualisierung

In [ ]:
def show(ax, img, title=None):
    '''Bild auf einer Achse anzeigen.'''
    ax.imshow(img)
    if title:
        ax.set_title(title)
    ax.axis('off')

def overlay_mask(image_np, mask, color=(255, 0, 0), alpha=0.5):
    '''Boolesche Maske mit Farbe über ein RGB-Bild legen.'''
    out = image_np.copy().astype(np.float32)
    for c in range(3):
        out[..., c] = np.where(
            mask, alpha * color[c] + (1 - alpha) * out[..., c], out[..., c]
        )
    return out.clip(0, 255).astype(np.uint8)

# Eine angenehme Farbpalette für Klassen / Instanzen
def get_color_palette(n=21, seed=42):
    rng = np.random.default_rng(seed)
    palette = (rng.random((n, 3)) * 255).astype(np.uint8)
    palette[0] = [0, 0, 0]  # Hintergrund schwarz
    return palette

## 2. Testbilder laden

Wir laden zwei Bilder von öffentlichen URLs: ein Wohnzimmer-Bild mit Katzen (klassisches Beispiel aus dem COCO-Datensatz) und ein Strassenbild.

In [ ]:
def load_image_from_url(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(io.BytesIO(response.content)).convert('RGB')

URL_CATS   = "http://images.cocodataset.org/val2017/000000039769.jpg"   # 2 Katzen auf Couch
URL_STREET = "http://images.cocodataset.org/val2017/000000000785.jpg"   # Strasse mit Personen

img_cats   = load_image_from_url(URL_CATS)
img_street = load_image_from_url(URL_STREET)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show(axes[0], img_cats,   f'Katzen   {img_cats.size}')
show(axes[1], img_street, f'Strasse  {img_street.size}')
plt.tight_layout(); plt.show()

### Eigenes Bild hochladen (optional)

Lade ein eigenes Bild hoch, wenn du es später mit den Modellen ausprobieren willst.

In [ ]:
my_image = None

if IN_COLAB:
    print("Bild auswählen (Cancel überspringt diesen Schritt):")
    try:
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            my_image = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')
            print(f"Geladen: {filename}, Grösse: {my_image.size}")
    except Exception as e:
        print(f"Kein Bild hochgeladen ({e}).")
else:
    # Lokal: Pfad anpassen, falls gewünscht
    # my_image = Image.open('mein_bild.jpg').convert('RGB')
    print("Lokale Umgebung: Pfad im Code anpassen, falls du ein eigenes Bild verwenden möchtest.")

if my_image is not None:
    plt.figure(figsize=(8, 6))
    plt.imshow(my_image); plt.axis('off'); plt.title('Dein Bild')
    plt.show()

---
## 3. Methode 1: FCN-ResNet50 – Semantic Segmentation

### Was passiert hier?

Wir verwenden ein **Fully Convolutional Network** (FCN) mit ResNet50 als Encoder. Das Modell wurde auf einem Subset von **COCO** mit 21 Klassen (20 + Hintergrund, identisch zu PASCAL VOC) trainiert.

Das Netz erhält ein Bild und gibt für **jeden Pixel** die wahrscheinlichste Klasse zurück. Mehrere Katzen werden als ein einziger Bereich "Katze" klassifiziert – Instanzen werden nicht unterschieden.

### Klassen, die das Modell kennt
*Hintergrund, Flugzeug, Fahrrad, Vogel, Boot, Flasche, Bus, Auto, Katze, Stuhl, Kuh, Esstisch, Hund, Pferd, Motorrad, Person, Topfpflanze, Schaf, Sofa, Zug, TV*

In [ ]:
from torchvision.models.segmentation import fcn_resnet50, FCN_ResNet50_Weights

# Modell + passende Vorverarbeitung laden
weights = FCN_ResNet50_Weights.DEFAULT
fcn_model = fcn_resnet50(weights=weights).eval().to(DEVICE)
fcn_preprocess = weights.transforms()
fcn_classes = weights.meta["categories"]

print(f"Anzahl Klassen: {len(fcn_classes)}")
print(f"Klassen: {fcn_classes}")

In [ ]:
def fcn_segment(pil_image):
    '''FCN auf ein PIL-Bild anwenden, Rückgabe: 2D-Array mit Klassenindex pro Pixel,
       reskaliert auf die Originalgrösse des Bildes.'''
    # Speichere die Originalgrösse des Bildes (Breite, Höhe)
    original_width, original_height = pil_image.size

    x = fcn_preprocess(pil_image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = fcn_model(x)["out"]
    # out: (1, num_classes, H_model_output, W_model_output) -> argmax über Klassenachse
    pred = out[0].argmax(dim=0).cpu().numpy()

    # Reskaliere die Vorhersagemaske zurück auf die Originalgrösse
    # Konvertiere NumPy-Array zu PIL Image für die Reskalierung
    pred_pil = Image.fromarray(pred.astype(np.uint8))
    # Verwende Nearest-Neighbor-Interpolation, um Klassen-IDs nicht zu verändern
    pred_resized = pred_pil.resize((original_width, original_height), Image.NEAREST)

    return np.array(pred_resized)

# Auf Katzenbild anwenden
pred_cats = fcn_segment(img_cats)

# Visualisieren
palette = get_color_palette(n=len(fcn_classes))
pred_rgb = palette[pred_cats]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
show(axes[0], img_cats, 'Original')
show(axes[1], pred_rgb, 'Semantic Mask (eingefärbt)')
overlay = overlay_mask(np.array(img_cats), pred_cats == fcn_classes.index('cat'), color=(255, 80, 80), alpha=0.6)
show(axes[2], overlay, 'Overlay: alle Katzen-Pixel')
plt.tight_layout(); plt.show()

# Welche Klassen wurden gefunden?
unique = np.unique(pred_cats)
print("Gefundene Klassen:")
for u in unique:
    pct = (pred_cats == u).mean() * 100
    print(f"  {fcn_classes[u]:15s}  {pct:5.1f} % der Pixel")

### 🎯 Beobachtung

**Schau dir das Overlay genau an. Wie viele Katzen siehst du im Original – und wie viele *separate* Bereiche hat die semantic Mask?**

In [ ]:
beobachtung_fcn = '''
Anzahl Katzen im Original: ???
Anzahl getrennter Bereiche in der Maske: ???

Was bedeutet das für die Aussagekraft semantic Segmentation?
???
'''
print(beobachtung_fcn)

<details>
<summary>💡 Erwartete Antwort</summary>

Im Original sind zwei Katzen, die sich berühren. Die Maske hat aber **einen** zusammenhängenden Bereich "Katze" – semantic Segmentation kann nicht zwischen den beiden Tieren unterscheiden. Wenn du die Tiere zählen oder jedem einzeln eine Eigenschaft zuordnen willst, brauchst du *Instance Segmentation*.

</details>

### 3.1 Auf das Strassenbild anwenden

Was findet das Modell auf dem zweiten Bild?

In [ ]:
pred_street = fcn_segment(img_street)
pred_rgb_street = palette[pred_street]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show(axes[0], img_street, 'Original')
show(axes[1], pred_rgb_street, 'Semantic Mask')
plt.tight_layout(); plt.show()

unique = np.unique(pred_street)
for u in unique:
    pct = (pred_street == u).mean() * 100
    if pct > 0.5:
        print(f"  {fcn_classes[u]:15s}  {pct:5.1f} %")

### 3.2 Eigenes Bild

Falls du oben ein Bild hochgeladen hast, probier es aus:

In [ ]:
if my_image is not None:
    pred_my = fcn_segment(my_image)
    pred_rgb_my = palette[pred_my]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    show(axes[0], my_image, 'Dein Bild')
    show(axes[1], pred_rgb_my, 'Semantic Mask')
    plt.tight_layout(); plt.show()

    unique = np.unique(pred_my)
    print("Gefundene Klassen:")
    for u in unique:
        pct = (pred_my == u).mean() * 100
        if pct > 0.5:
            print(f"  {fcn_classes[u]:15s}  {pct:5.1f} %")
    print("\n👉 Wichtig: Das Modell kennt nur die 21 Klassen oben. Alles andere wird zu 'background'.")
else:
    print("Kein eigenes Bild geladen.")

---
## 4. Methode 2: Mask R-CNN – Instance Segmentation

### Was ist anders?

Mask R-CNN baut auf **Faster R-CNN** auf (Object Detection) und ergänzt einen **Mask Head**, der für jede gefundene Bounding Box eine pixelgenaue Maske vorhersagt.

Das bedeutet: pro **Instanz** eine eigene Maske. Drei Katzen → drei Masken. Trainiert auf **COCO** mit 80 Klassen.

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
mrcnn_model = maskrcnn_resnet50_fpn(weights=weights).eval().to(DEVICE)
mrcnn_preprocess = weights.transforms()
mrcnn_classes = weights.meta["categories"]

print(f"Anzahl Klassen: {len(mrcnn_classes)}")
print(f"Beispiel-Klassen: {mrcnn_classes[1:11]}")

In [ ]:
def mrcnn_segment(pil_image, score_threshold=0.7):
    '''Mask R-CNN anwenden, Filter nach Konfidenz, Rückgabe: dict mit boxes/labels/scores/masks.'''
    x = mrcnn_preprocess(pil_image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        raw = mrcnn_model(x)[0]
    keep = raw['scores'] > score_threshold
    return {
        'boxes':  raw['boxes'][keep].cpu().numpy(),
        'labels': raw['labels'][keep].cpu().numpy(),
        'scores': raw['scores'][keep].cpu().numpy(),
        # masks sind Wahrscheinlichkeiten (0..1), wir thresholden bei 0.5
        'masks':  (raw['masks'][keep, 0] > 0.5).cpu().numpy(),
    }

def visualize_instances(pil_image, result, class_names):
    '''Jede Instanz mit eigener Farbe überlagern und Label anschreiben.'''
    img_np = np.array(pil_image)
    out = img_np.copy()
    inst_palette = get_color_palette(n=max(20, len(result['masks']) + 1), seed=7)

    fig, ax = plt.subplots(figsize=(10, 8))
    for i, (mask, label, score, box) in enumerate(zip(
        result['masks'], result['labels'], result['scores'], result['boxes']
    )):
        color = tuple(int(c) for c in inst_palette[i + 1])
        out = overlay_mask(out, mask, color=color, alpha=0.5)
        x1, y1, x2, y2 = box
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                    fill=False, edgecolor=np.array(color)/255, linewidth=2))
        ax.text(x1, y1 - 4, f"{class_names[label]} {score:.2f}",
                color='white', fontsize=10,
                bbox=dict(facecolor=np.array(color)/255, alpha=0.8, edgecolor='none', pad=2))
    ax.imshow(out); ax.axis('off')
    ax.set_title(f"{len(result['masks'])} Instanzen erkannt (Score > {0.7})")
    plt.tight_layout(); plt.show()

# Auf Katzenbild
result_cats = mrcnn_segment(img_cats, score_threshold=0.7)
visualize_instances(img_cats, result_cats, mrcnn_classes)
print(f"Gefundene Instanzen: {len(result_cats['masks'])}")
for label, score in zip(result_cats['labels'], result_cats['scores']):
    print(f"  {mrcnn_classes[label]:15s} (Score {score:.2f})")

### 4.1 Konfidenzschwelle anpassen (Aufgabe)

Der `score_threshold` steuert, wie sicher das Modell sein muss, bevor es eine Vorhersage als gültig akzeptiert. Probier verschiedene Werte:

In [ ]:
# AUFGABE: Was passiert bei sehr tiefem (0.3) und sehr hohem (0.95) Threshold?
# Wende Mask R-CNN mit drei verschiedenen Thresholds auf das Strassenbild an.

thresholds = [???, ???, ???]  # z.B. [0.3, 0.7, 0.95]

for th in thresholds:
    print(f"\n--- Threshold = {th} ---")
    result = mrcnn_segment(img_street, score_threshold=th)
    visualize_instances(img_street, result, mrcnn_classes)

<details>
<summary>💡 Erwartetes Verhalten</summary>

- **Tiefer Threshold (0.3)**: Viele Detections, auch falsche – das Bild wird "verrauscht", Mehrfacherkennungen überlagern sich.
- **Mittlerer Threshold (0.7)**: Ausgewogenes Verhältnis von Recall und Precision.
- **Hoher Threshold (0.95)**: Nur sehr sichere Erkennungen – manche echten Objekte fehlen.

**Take-away:** Der Threshold ist ein Hyperparameter, den du je nach Anwendung wählst. Bei Sicherheit-relevanten Systemen (Medizin, autonomes Fahren) eher hoch, bei explorativer Suche eher tief.

</details>

### 4.2 Eigenes Bild

In [ ]:
if my_image is not None:
    result_my = mrcnn_segment(my_image, score_threshold=0.7)
    visualize_instances(my_image, result_my, mrcnn_classes)
else:
    print("Kein eigenes Bild geladen.")

---
## 5. Methode 3: Segment Anything Model (SAM)

SAM (Meta AI, 2023) ist ein **Foundation Model** für Segmentierung. Du gibst ihm:
- ein Bild **und**
- einen **Prompt** (Punkt, Box oder mehrere Punkte)

→ und es liefert dir eine Maske rund um das Objekt am Prompt-Ort. **Kein Training nötig**, kein Klassen-Label – SAM segmentiert *was du willst*, nicht was es kennt.

> 💡 **Geduld:** Das Laden des Modells dauert ein paar Sekunden (~375 MB), und auf CPU dauert eine Inferenz ~30s. Mit GPU sind es ~1-2s.

In [ ]:
from transformers import SamModel, SamProcessor

sam_processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
sam_model = SamModel.from_pretrained("facebook/sam-vit-base").eval().to(DEVICE)
print("SAM geladen.")

In [ ]:
def sam_predict_point(pil_image, x, y):
    '''SAM mit einem Punkt-Prompt.'''
    input_points = [[[x, y]]]  # eine Liste mit einem Punkt
    inputs = sam_processor(pil_image, input_points=input_points, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = sam_model(**inputs, multimask_output=True)
    # post_process_masks rechnet zurück auf Originalgrösse
    masks = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    # masks[0]: shape (1, 3, H, W) - SAM liefert 3 Kandidaten unterschiedlicher Granularität
    scores = outputs.iou_scores[0, 0].cpu().numpy()
    return masks[0][0].numpy().astype(bool), scores

def visualize_sam(pil_image, masks, scores, point=None, box=None):
    img_np = np.array(pil_image)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for i in range(3):
        overlay = overlay_mask(img_np, masks[i], color=(80, 200, 255), alpha=0.55)
        axes[i].imshow(overlay)
        if point is not None:
            axes[i].plot(point[0], point[1], 'r*', markersize=18, markeredgecolor='white', markeredgewidth=1.5)
        if box is not None:
            x1, y1, x2, y2 = box
            axes[i].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                             fill=False, edgecolor='red', linewidth=2))
        axes[i].set_title(f"Maske {i+1} – IoU-Score {scores[i]:.2f}")
        axes[i].axis('off')
    plt.tight_layout(); plt.show()

# Punkt auf eine Katze setzen (Bildkoordinaten anschauen!)
print(f"Bildgrösse: {img_cats.size}")  # (Breite, Höhe)
masks, scores = sam_predict_point(img_cats, x=450, y=300)
visualize_sam(img_cats, masks, scores, point=(450, 300))

### Drei Masken? Warum?

SAM liefert pro Prompt **drei Kandidaten** mit unterschiedlicher Granularität:
- die kleinste Maske (z. B. nur das Auge der Katze),
- eine mittlere (z. B. der Kopf),
- die grösste (z. B. die ganze Katze).

Das ist nützlich, weil ein Punkt mehrdeutig sein kann ("meinst du das Detail oder das ganze Objekt?").

### 5.1 Andere Punkte ausprobieren (Aufgabe)

Probier verschiedene Punkte – auch im Hintergrund. Was passiert?

In [ ]:
# AUFGABE: Setze einen Punkt auf die andere Katze, einen auf das Sofa
#          und einen auf den Hintergrund. Was beobachtest du?

# Hinweis: img_cats hat Grösse 640 x 480 (Breite x Höhe)
points_to_try = [
    (???, ???),    # auf die andere Katze
    (???, ???),    # auf das Sofa
    (???, ???),    # auf den Hintergrund
]

for px, py in points_to_try:
    print(f"\nPrompt: ({px}, {py})")
    masks, scores = sam_predict_point(img_cats, x=px, y=py)
    visualize_sam(img_cats, masks, scores, point=(px, py))

### 5.2 Box-Prompt

Statt eines Punkts kannst du auch eine **Bounding Box** angeben. Das ist oft präziser, weil SAM weiss, *wo* das Objekt liegt.

In [ ]:
def sam_predict_box(pil_image, x1, y1, x2, y2):
    input_boxes = [[[x1, y1, x2, y2]]]
    inputs = sam_processor(pil_image, input_boxes=input_boxes, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = sam_model(**inputs, multimask_output=False)
    masks = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    return masks[0][0, 0].numpy().astype(bool)

# Box um die linke Katze
mask = sam_predict_box(img_cats, x1=20, y1=80, x2=320, y2=480)

img_np = np.array(img_cats)
overlay = overlay_mask(img_np, mask, color=(80, 200, 255), alpha=0.55)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(overlay)
ax.add_patch(plt.Rectangle((20, 80), 320-20, 480-80, fill=False, edgecolor='red', linewidth=2))
ax.set_title('SAM mit Box-Prompt'); ax.axis('off')
plt.tight_layout(); plt.show()

### 5.3 Eigenes Bild + eigener Prompt

In [ ]:
if my_image is not None:
    print(f"Bildgrösse deines Bildes: {my_image.size}")
    # AUFGABE: Setze sinnvolle Koordinaten für dein Bild
    px, py = ???, ???
    masks, scores = sam_predict_point(my_image, x=px, y=py)
    visualize_sam(my_image, masks, scores, point=(px, py))
else:
    print("Kein eigenes Bild geladen.")

---
## 6. Direktvergleich der drei Methoden

Wir lassen alle drei Modelle auf demselben Bild laufen und stellen die Ergebnisse nebeneinander.

In [ ]:
# Alle drei Modelle auf das Katzenbild anwenden

# 1) FCN
pred_fcn = fcn_segment(img_cats)
mask_fcn_cats = pred_fcn == fcn_classes.index('cat')

# 2) Mask R-CNN
res_mrcnn = mrcnn_segment(img_cats, score_threshold=0.7)
# Alle Masken kombinieren, jede in eigener Farbe
img_np = np.array(img_cats)
overlay_mrcnn = img_np.copy()
inst_palette = get_color_palette(n=10, seed=7)
for i, m in enumerate(res_mrcnn['masks']):
    overlay_mrcnn = overlay_mask(overlay_mrcnn, m, color=tuple(int(c) for c in inst_palette[i + 1]), alpha=0.5)

# 3) SAM (Box um beide Katzen)
mask_sam = sam_predict_box(img_cats, x1=20, y1=80, x2=620, y2=480)
overlay_sam = overlay_mask(img_np, mask_sam, color=(80, 200, 255), alpha=0.55)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
show(axes[0], img_cats, 'Original')
show(axes[1], overlay_mask(img_np, mask_fcn_cats, color=(255, 80, 80), alpha=0.55),
     f'FCN (semantic)\nalle "cat"-Pixel')
show(axes[2], overlay_mrcnn, f'Mask R-CNN (instance)\n{len(res_mrcnn["masks"])} Instanzen')
show(axes[3], overlay_sam, 'SAM (promptbar)\nBox-Prompt um beide Katzen')
plt.tight_layout(); plt.show()

## 7. Zusammenfassung & Reflexion

### Stark vereinfacht

| Methode | Eingabe | Output | Klassen-Wissen |
|---|---|---|---|
| **FCN** | Bild | Klasse pro Pixel | 21 fixe Klassen |
| **Mask R-CNN** | Bild | Maske pro Instanz + Klasse + Score | 80 fixe Klassen |
| **SAM** | Bild + Prompt | Maske rund um Prompt | keine – nur "Objekt" |

### 🎯 Abschlussreflexion

Beantworte für dich:

1. Welches Modell hat dich bei deinem eigenen Bild am meisten überzeugt?
2. Wann würdest du **FCN** wählen, wann **Mask R-CNN**, wann **SAM**?
3. Was ist die Gemeinsamkeit aller drei – im Vergleich zu den klassischen Methoden aus dem letzten Notebook?